# Training Notebook

## Import of all needed scripts

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys

sys.path.insert(0, "../../src")

import gc
import time

import numpy as np
import torch
import torch.distributed as dist

from juart.dl.checkpoint.manager import CheckpointManager
from juart.dl.data.training import DatasetTraining
from juart.dl.loss.loss import JointLoss
from juart.dl.model.unrollnet import (
    LookaheadModel,
    UnrolledNet,
)
from juart.dl.operation.modules import training, validation
from juart.dl.utils.dist import GradientAccumulator

## Defining shuffle function
When training a model the slices and subjects of the datasets should not be in order. Therefore this function creates a random order for every single epoch, granting that every sclice is used once and only once in every epoch. It is possible to give the function a seed so that the random order is the same order in every training. This is used for better comparability of the models, because the accuracy of the models can slightly variate with the order of the slices and subjects in the training.

In [ ]:
def shuffled_indices(num_samples, num_epochs, rng):
    indices = np.repeat(np.arange(num_samples), num_epochs)
    indices = indices.reshape((num_samples, num_epochs))
    indices = rng.permuted(indices, axis=0)
    indices = indices.T.ravel()

    # Check if each sample is used once and only once in every epoch
    assert indices.size == num_samples * num_epochs
    for i in np.split(indices, num_epochs):
        assert np.unique(i).size == num_samples

    return indices

## Define important variables

In [ ]:
nX, nY = 256, 256  # Number of pixels in x-/y-direction
nTI, nTE = 2, 2  # Number of measurements during the T1/T2 decay
device = "cpu"  # defines whether the model should be trained on the cpu or gpu
group = None  # ?
group_rank = 0  # ?
group_index = 0  # ?

num_epochs = 1  # Number of epochs of the training
num_groups = 1  # ?
batch_size = 1  # Number of slices that should be used for training per batch
nD = 1  # Number of subjects
nS = 160  # Number of slices per subject

batch_size_local = batch_size // num_groups

num_iterations = nD * nS * num_epochs

## Randomize indices
The shuffle function gets used to shuffle the indices of the training and validation dataset.

In [ ]:
rng = np.random.default_rng(seed=0)

training_indices = shuffled_indices(nD * nS, num_epochs, rng)
training_indices_batched = training_indices.reshape((-1, batch_size_local, num_groups))

validation_indices = shuffled_indices(nD * nS, num_epochs, rng)
validation_indices_batched = validation_indices.reshape(
    (-1, batch_size_local, num_groups)
)

In [ ]:
dist.init_process_group(
    backend="gloo", init_method="tcp://127.0.0.1:23456", world_size=1, rank=0
)

In [ ]:
model = UnrolledNet(
    (nX, nY),
    contrasts=nTI * nTE,
    features=64,
    CG_Iter=10,
    num_unroll_blocks=10,
    activation="ReLU",
    disable_progress_bar=True,
    timing_level=0,
    validation_level=0,
    device=device,
)

In [ ]:
loss_fn = JointLoss(
    (nX, nY),
    (3, 3),
    weights_kspace_loss=(0.5, 0.5),
    weights_ispace_loss=(0.0, 0.0),
    weights_wavelet_loss=(0.0, 0.0),
    weights_hankel_loss=(0.0, 0.0),
    weights_casorati_loss=(0.0, 0.0),
    normalized_loss=True,
    timing_level=0,
    validation_level=0,
    group=group,
    device=device,
)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001,
    betas=[0.9, 0.999],
    eps=1.0e-8,
    weight_decay=0.0,
)

In [ ]:
accumulator = GradientAccumulator(
    model,
    accumulation_steps=batch_size_local,
    max_norm=1.0,
    normalized_gradient=False,
)

In [ ]:
averaged_model = LookaheadModel(
    model,
    alpha=0.5,
    k=5,
)

In [ ]:
checkpoint_manager = CheckpointManager(
    directory="model_8spokes",
    root_dir="/home/jovyan/models",
    backend="local",
)

In [ ]:
load_model_state = True
load_averaged_model_state = True
load_optim_state = True
load_metrics = True

In [ ]:
if load_model_state:
    print("Loading model state ...")
    checkpoint = checkpoint_manager.load(["model_state"], map_location=device)
    if all(checkpoint.values()):
        model.load_state_dict(checkpoint["model_state"])
    else:
        print("Could not load model state.")

In [ ]:
if load_averaged_model_state:
    print("Loading averaged model state ...")
    checkpoint = checkpoint_manager.load(["averaged_model_state"], map_location=device)
    if all(checkpoint.values()):
        averaged_model.load_state_dict(checkpoint["averaged_model_state"])
    else:
        print("Could not load averaged model state.")

In [ ]:
if load_optim_state:
    print("Loading optim state ...")
    checkpoint = checkpoint_manager.load(["optim_state"], map_location=device)
    if all(checkpoint.values()):
        optimizer.load_state_dict(checkpoint["optim_state"])
    else:
        print("Could not load optim state.")

In [ ]:
total_trn_loss = list()
total_val_loss = list()
iteration = 0

In [ ]:
if load_metrics:
    print("Loading metrics ...")
    checkpoint = checkpoint_manager.load(["trn_loss", "val_loss", "iteration"])
    if all(checkpoint.values()):
        total_trn_loss = list(checkpoint["trn_loss"])
        total_val_loss = list(checkpoint["val_loss"])
        iteration = checkpoint["iteration"]
    else:
        print("Could not load metrics.")

In [ ]:
print(f"Continue with iteration {iteration} ...")

In [ ]:
training_data = DatasetTraining(
    "qrage/sessions/%s/preproc.zarr/preproc.zarr",
    ["7T1566"],
    np.arange(0, 160),
    8,
    [0.0, 0.5, 0.5],
    mode="training",
    group_rank=group_rank,
    endpoint_url="https://s3.fz-juelich.de",
    backend="s3",
)

In [ ]:
validation_data = DatasetTraining(
    "qrage/sessions/%s/preproc.zarr/preproc.zarr",
    ["7T1566"],
    np.arange(0, 160),
    8,
    [0.0, 0.5, 0.5],
    mode="validation",
    group_rank=group_rank,
    endpoint_url="https://s3.fz-juelich.de",
    backend="s3",
)

In [ ]:
while iteration < num_iterations:
    tic = time.time()

    # Reset the seed so that training can be resumed
    np.random.seed(iteration)
    torch.manual_seed(iteration)

    training_index = training_indices_batched[
        iteration // batch_size,
        :,
        group_index,
    ].tolist()
    validation_index = validation_indices_batched[
        iteration // batch_size, :, group_index
    ].tolist()

    if True:  # options["model_training"]:
        print(f"Training index {training_index} ...")

        trn_loss = training(
            training_index,
            training_data,
            model,
            loss_fn,
            optimizer,
            accumulator,
            group=group,
            device=device,
        )

        averaged_model.update_parameters(
            model,
        )

        torch.cuda.empty_cache()
        gc.collect()

    else:
        trn_loss = [0] * batch_size

    if False:  # options["model_validation"]:
        print(f"Validation index {validation_index} ...")

        val_loss = validation(
            validation_index,
            validation_data,
            averaged_model,
            loss_fn,
            group=group,
            device=device,
        )
        torch.cuda.empty_cache()
        gc.collect()

    else:
        val_loss = [0] * batch_size

    total_trn_loss += trn_loss
    total_val_loss += val_loss

    # Completed epoch
    if (
        True  # options["save_checkpoint"]
        and np.mod(iteration + batch_size, nD * nS) == 0
    ):
        print("Creating tagged checkpoint ...")

        checkpoint = {
            "iteration": iteration + batch_size,
            "model_state": model.state_dict(),
            "averaged_model_state": averaged_model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "trn_loss": total_trn_loss,
            "val_loss": total_val_loss,
        }

        epoch = (iteration + batch_size) // (nD * nS)
        checkpoint_manager.save(checkpoint, tag=f"_epoch_{epoch}")

        if True:  # options["single_epoch"]
            # Also save the checkpoint as untagged checkpoint
            # Otherwise, training will be stuck in endless loop
            checkpoint_manager.save(checkpoint)
            checkpoint_manager.release()
            break

    # Intermediate checkpoint
    elif (
        True  # options["save_checkpoint"]
        and np.mod(iteration + batch_size, 10)
        == 0  # 10 = options["checkpoint_frequency"]
    ):
        print("Creating untagged checkpoint ...")

        checkpoint = {
            "iteration": iteration + batch_size,
            "model_state": model.state_dict(),
            "averaged_model_state": averaged_model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "trn_loss": total_trn_loss,
            "val_loss": total_val_loss,
        }

        checkpoint_manager.save(checkpoint, block=False)

    toc = time.time() - tic

    print(
        (
            f"Iteration: {iteration} - "
            + f"Elapsed time: {toc:.0f} - "
            + f"Training loss: {[f'{loss:.3f}' for loss in trn_loss]} - "
            + f"Validation loss: {[f'{loss:.3f}' for loss in val_loss]}"
        )
    )

    torch.cuda.empty_cache()
    gc.collect()

    iteration += batch_size